# Senior Data Engineer – Advanced ETL Pipeline

**Assessment basis:** `TechnicalAssessment 5 (1).pdf`, `sales_data 2.csv`, and `product_reference 2.csv`.

This notebook is designed for **Azure Synapse Spark / PySpark**. It implements:
- null handling
- exact duplicate removal
- data validation
- product-reference lookup
- external exchange-rate API integration
- cached/default rate fallback
- USD conversion
- conversion logging
- detailed error logging
- rejected-record archival
- rejection-rate monitoring with a hard **5% failure threshold**
- load to Azure SQL Database / SQL Server

> Important: the supplied sample contains deliberate quality issues. With the validation rules below, the sample is expected to exceed the 5% rejection threshold, so a production run should archive/log the bad records and **fail before publishing the target dataset**.


In [ ]:
# 1. Imports and configuration
from pyspark.sql import functions as F, types as T
from pyspark.sql.window import Window
from datetime import datetime, timezone
import json
import requests
import uuid

try:
    from notebookutils import mssparkutils
except Exception:
    mssparkutils = None

BATCH_ID = str(uuid.uuid4())
RUN_TS_UTC = datetime.now(timezone.utc).isoformat()

# Replace these placeholders with environment-specific values.
RAW_SALES_PATH = "abfss://<container>@<storage-account>.dfs.core.windows.net/bronze/sales_data.csv"
RAW_PRODUCT_PATH = "abfss://<container>@<storage-account>.dfs.core.windows.net/bronze/product_reference.csv"
REJECT_PATH = "abfss://<container>@<storage-account>.dfs.core.windows.net/rejected/sales/"
ERROR_LOG_PATH = "abfss://<container>@<storage-account>.dfs.core.windows.net/logs/etl_errors/"
CONVERSION_LOG_PATH = "abfss://<container>@<storage-account>.dfs.core.windows.net/logs/currency_conversions/"
CACHE_PATH = "abfss://<container>@<storage-account>.dfs.core.windows.net/reference/exchange_rates/latest.json"

EXCHANGE_API_URL = "https://api.exchangerate-api.com/v4/latest/EUR"
REJECTION_THRESHOLD = 0.05

# Fallback values are deliberately configurable. In production, use approved
# cached rates from the last successful API response when available.
DEFAULT_RATES_TO_USD = {
    "USD": 1.0,
    "EUR": 1.08,
    "GBP": 1.27
}


In [ ]:
# 2. Explicit source schemas
sales_schema = T.StructType([
    T.StructField("OrderID", T.LongType(), True),
    T.StructField("ProductID", T.StringType(), True),
    T.StructField("SaleAmount", T.DecimalType(18, 2), True),
    T.StructField("OrderDate", T.StringType(), True),
    T.StructField("Region", T.StringType(), True),
    T.StructField("CustomerID", T.StringType(), True),
    T.StructField("Discount", T.DecimalType(8, 4), True),
    T.StructField("Currency", T.StringType(), True),
])

product_schema = T.StructType([
    T.StructField("ProductID", T.StringType(), True),
    T.StructField("ProductName", T.StringType(), True),
    T.StructField("Category", T.StringType(), True),
])

sales_raw = (
    spark.read
    .option("header", True)
    .schema(sales_schema)
    .csv(RAW_SALES_PATH)
)

products = (
    spark.read
    .option("header", True)
    .schema(product_schema)
    .csv(RAW_PRODUCT_PATH)
)

print(f"Batch ID: {BATCH_ID}")
print(f"Input rows: {sales_raw.count()}")
print(f"Product reference rows: {products.count()}")


In [ ]:
# 3. Standardization and null handling
sales_std = (
    sales_raw
    .select(
        F.col("OrderID"),
        F.upper(F.trim(F.col("ProductID"))).alias("ProductID"),
        F.col("SaleAmount"),
        F.trim(F.col("OrderDate")).alias("OrderDateRaw"),
        F.initcap(F.trim(F.col("Region"))).alias("Region"),
        F.when(F.trim(F.col("CustomerID")) == "", None)
         .otherwise(F.trim(F.col("CustomerID"))).alias("CustomerID"),
        F.col("Discount"),
        F.upper(F.trim(F.col("Currency"))).alias("Currency")
    )
    # Non-critical nulls are defaulted.
    .withColumn("CustomerID", F.coalesce(F.col("CustomerID"), F.lit("UNKNOWN")))
    .withColumn("Discount", F.coalesce(F.col("Discount"), F.lit(0).cast(T.DecimalType(8,4))))
    # Parse the mixed formats present in the supplied data.
    .withColumn(
        "OrderDate",
        F.coalesce(
            F.to_date("OrderDateRaw", "dd/MM/yyyy"),
            F.to_date("OrderDateRaw", "dd-MM-yyyy"),
            F.to_date("OrderDateRaw", "yyyy-MM-dd")
        )
    )
)

# Exact duplicate removal; the supplied sample contains one exact duplicate.
sales_dedup = sales_std.dropDuplicates()

print("Rows after exact deduplication:", sales_dedup.count())


In [ ]:
# 4. Product lookup / enrichment
products_clean = (
    products
    .select(
        F.upper(F.trim(F.col("ProductID"))).alias("ProductID"),
        F.trim(F.col("ProductName")).alias("ProductName"),
        F.trim(F.col("Category")).alias("Category")
    )
    .dropDuplicates(["ProductID"])
)

enriched_lookup = (
    sales_dedup.alias("s")
    .join(F.broadcast(products_clean).alias("p"), on="ProductID", how="left")
)


In [ ]:
# 5. Data validation and error classification
# Assumptions for this assessment:
# - SaleAmount is a monetary sale value and must be non-null and >= 0.
# - OrderDate must be present and parseable.
# - ProductID must resolve in the reference table.
# - Region and Currency must be populated.
# - CustomerID and Discount are non-critical and are defaulted above.

validated = (
    enriched_lookup
    .withColumn(
        "ErrorArray",
        F.array_remove(
            F.array(
                F.when(F.col("OrderID").isNull() | (F.col("OrderID") <= 0), F.lit("INVALID_ORDER_ID")),
                F.when(F.col("SaleAmount").isNull(), F.lit("NULL_SALE_AMOUNT")),
                F.when(F.col("SaleAmount") < 0, F.lit("NEGATIVE_SALE_AMOUNT")),
                F.when(F.col("OrderDate").isNull(), F.lit("INVALID_OR_NULL_ORDER_DATE")),
                F.when(F.col("ProductName").isNull(), F.lit("PRODUCT_LOOKUP_FAILED")),
                F.when(F.col("Region").isNull() | (F.trim(F.col("Region")) == ""), F.lit("NULL_REGION")),
                F.when(F.col("Currency").isNull() | (F.trim(F.col("Currency")) == ""), F.lit("NULL_CURRENCY"))
            ),
            None
        )
    )
    .withColumn("IsRejected", F.size("ErrorArray") > 0)
    .withColumn("ErrorType", F.concat_ws("|", "ErrorArray"))
    .withColumn("ErrorMessage", F.concat_ws("; ", "ErrorArray"))
)

rejected_df = (
    validated.filter("IsRejected")
    .withColumn("BatchID", F.lit(BATCH_ID))
    .withColumn("ErrorTimestamp", F.current_timestamp())
    .select(
        "BatchID","OrderID","ProductID","SaleAmount","OrderDateRaw","OrderDate",
        "Region","CustomerID","Discount","Currency","ErrorType","ErrorMessage","ErrorTimestamp"
    )
)

valid_df = validated.filter(~F.col("IsRejected")).drop("ErrorArray","IsRejected","ErrorType","ErrorMessage")

rejected_count = rejected_df.count()
input_count_after_dedup = sales_dedup.count()
rejection_rate = rejected_count / input_count_after_dedup if input_count_after_dedup else 0.0

print(f"Deduplicated input: {input_count_after_dedup}")
print(f"Rejected: {rejected_count}")
print(f"Rejection rate: {rejection_rate:.2%}")


In [ ]:
# 6. Exchange-rate API with cache/default fallback
def _read_text(path):
    if mssparkutils is None:
        return None
    try:
        return mssparkutils.fs.head(path, 1024 * 1024)
    except Exception:
        return None

def _write_text(path, text):
    if mssparkutils is None:
        return False
    try:
        mssparkutils.fs.put(path, text, True)
        return True
    except Exception:
        return False

def fetch_rates_to_usd():
    # USD is the identity rate.
    fallback = dict(DEFAULT_RATES_TO_USD)
    try:
        response = requests.get(EXCHANGE_API_URL, timeout=10)
        response.raise_for_status()
        payload = response.json()

        if payload.get("result") == "success" or "rates" in payload:
            rates = payload["rates"]
            eur_to_usd = float(rates["USD"])
            gbp_per_eur = float(rates["GBP"])

            # Endpoint base is EUR:
            # EUR -> USD = USD rate
            # GBP -> USD = (USD per EUR) / (GBP per EUR)
            current = {
                "USD": 1.0,
                "EUR": eur_to_usd,
                "GBP": eur_to_usd / gbp_per_eur
            }

            cache_payload = {
                "fetched_at_utc": datetime.now(timezone.utc).isoformat(),
                "source": EXCHANGE_API_URL,
                "rates_to_usd": current
            }
            _write_text(CACHE_PATH, json.dumps(cache_payload))
            return current, "API"
    except Exception as api_error:
        print(f"Exchange API failure: {type(api_error).__name__}: {api_error}")

    # Cached fallback
    try:
        cached = _read_text(CACHE_PATH)
        if cached:
            payload = json.loads(cached)
            cached_rates = payload.get("rates_to_usd")
            if cached_rates and all(k in cached_rates for k in ("USD","EUR","GBP")):
                return {k: float(v) for k, v in cached_rates.items()}, "CACHE"
    except Exception as cache_error:
        print(f"Cache read failure: {type(cache_error).__name__}: {cache_error}")

    return fallback, "DEFAULT"

rates_to_usd, rate_source = fetch_rates_to_usd()
print("Rate source:", rate_source)
print("Rates to USD:", rates_to_usd)


In [ ]:
# 7. Convert valid SaleAmount values to USD
rates_map = spark.sparkContext.broadcast(rates_to_usd)

@F.udf(returnType=T.DoubleType())
def get_rate_to_usd(currency):
    if currency is None:
        return None
    return float(rates_map.value.get(currency)) if currency in rates_map.value else None

converted = (
    valid_df
    .withColumn("ConversionRateToUSD", get_rate_to_usd("Currency"))
    .withColumn(
        "SaleAmountUSD",
        F.round(F.col("SaleAmount").cast("double") * F.col("ConversionRateToUSD"), 2)
    )
    .withColumn("ConversionTimestamp", F.current_timestamp())
    .withColumn("RateSource", F.lit(rate_source))
    .withColumn("BatchID", F.lit(BATCH_ID))
    .withColumn("LoadTimestamp", F.current_timestamp())
)

# If a currency appears that is not supported by the configured rate set,
# classify it as an error rather than silently multiplying by a null rate.
unknown_rate_df = converted.filter(F.col("ConversionRateToUSD").isNull())
if unknown_rate_df.limit(1).count() > 0:
    raise RuntimeError("Unsupported currency encountered; conversion rate missing.")


In [ ]:
# 8. Conversion audit log
conversion_log_df = (
    converted
    .select(
        "BatchID","OrderID","ProductID","Currency",
        "SaleAmount","ConversionRateToUSD","SaleAmountUSD",
        "ConversionTimestamp","RateSource"
    )
    .withColumn(
        "RecordInfo",
        F.to_json(
            F.struct(
                "OrderID","ProductID","Currency","SaleAmount","SaleAmountUSD"
            )
        )
    )
)

# Persist as Delta/Parquet in ADLS; the SQL target can be loaded afterward.
conversion_log_df.write.mode("append").format("parquet").save(CONVERSION_LOG_PATH)


In [ ]:
# 9. Rejected-record archive and detailed error logging
error_log_df = (
    rejected_df
    .select(
        "BatchID","OrderID","ProductID","ErrorType","ErrorMessage","ErrorTimestamp",
        F.to_json(
            F.struct(
                "OrderID","ProductID","SaleAmount","OrderDateRaw",
                "Region","CustomerID","Discount","Currency"
            )
        ).alias("RecordDetails")
    )
)

rejected_df.write.mode("append").format("parquet").save(REJECT_PATH)
error_log_df.write.mode("append").format("parquet").save(ERROR_LOG_PATH)


In [ ]:
# 11. Azure SQL load (execute only after the quality gate passes)
# Prefer Synapse/ADF linked-service authentication in production. The JDBC function below
# is intentionally parameterized so credentials do not live in source control.

def load_to_sql(df, jdbc_url, target_table, connection_properties, mode="append"):
    required = {"driver", "user", "password"}
    missing = required - set(connection_properties)
    if missing:
        raise ValueError(f"Missing JDBC connection properties: {sorted(missing)}")

    (df.write
       .mode(mode)
       .jdbc(jdbc_url, target_table, properties=connection_properties))

# Example configuration (replace placeholders using Key Vault / linked service):
# jdbc_url = "jdbc:sqlserver://<server>.database.windows.net:1433;database=<db>"
# connection_props = {
#     "user": "<managed-identity-or-service-principal-user>",
#     "password": "<secret-injected-at-runtime>",
#     "driver": "com.microsoft.sqlserver.jdbc.SQLServerDriver"
# }
# load_to_sql(target_df, jdbc_url, "dbo.SalesEnriched", connection_props)

print("Quality gate passed; target_df is ready for Azure SQL load.")


## Sample-data outcome

For the supplied 20-row file:
- 1 exact duplicate is removed.
- 19 rows remain after exact deduplication.
- 6 rows fail the critical validation rules used in this notebook.
- Rejection rate = **6 / 19 = 31.58%**, which is above the required **5%** threshold.
- Therefore, the expected sample execution is a **controlled job failure after rejected-record archival/error logging and before target publication**.

This behavior is intentional and directly implements the assessment's quality-gate requirement.
